# 02 - Benford's Law Anomaly Detection

**PolyWatch — Market Integrity Monitoring System**  
**Member C — Core Algorithm Module**

This notebook demonstrates the Benford's Law detector for identifying
potentially fabricated trading data by checking first-digit distributions.

## Key Concepts
- Benford's Law: Leading digit d appears with probability P(d) = log10(1 + 1/d)
- Chi-squared goodness-of-fit test
- KS test
- MAD (Mean Absolute Deviation) conformity metric
- Sliding window temporal analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from core_analysis.benford_detector import (
    BenfordDetector, BenfordConfig, BENFORD_PROBS,
    prepare_price_changes, run_benford_analysis,
)
from core_analysis.db_interface import get_price_series

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports OK')

## 1. Benford's Law Theoretical Distribution

In [ ]:
digits = list(range(1, 10))
probs = [BENFORD_PROBS[d] for d in digits]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(digits, probs, color='steelblue', alpha=0.8, edgecolor='navy')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{p:.1%}', ha='center', fontsize=10)
ax.set_xlabel('First Digit', fontsize=12)
ax.set_ylabel('Expected Probability', fontsize=12)
ax.set_title("Benford's Law: Expected First Digit Distribution", fontsize=14)
ax.set_xticks(digits)
plt.tight_layout()
plt.show()

## 2. Load and Prepare Data

In [ ]:
# Load election data
slug = 'presidential-election-winner-2024'
price_df = get_price_series(slug)
price_series = price_df['price']

# Prepare price changes as proxy for trade data
price_changes = prepare_price_changes(price_series)

print(f'Market: {slug}')
print(f'Raw prices: {len(price_series)}')
print(f'Price changes (non-zero): {len(price_changes)}')
print(f'Price changes range: {price_changes.min():.2f} ~ {price_changes.max():.2f}')

## 3. Full Benford Analysis

In [ ]:
detector = BenfordDetector()
analysis = detector.analyze(price_changes)

print('=== Benford Analysis Results ===')
print(f'Valid samples: {analysis["n_valid"]} / {analysis["n_total"]}')
print(f'Overall conforming: {analysis["overall_conforming"]}')
print(f'\nChi-squared test:')
for k, v in analysis['chi_squared'].items():
    print(f'  {k}: {v}')
print(f'\nKS test:')
for k, v in analysis['ks_test'].items():
    print(f'  {k}: {v}')
print(f'\nMAD test:')
for k, v in analysis['mad_test'].items():
    print(f'  {k}: {v}')

## 4. Observed vs Expected Distribution

In [ ]:
observed = analysis['observed_distribution']
expected = analysis['expected_distribution']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, 10)
width = 0.35

bars1 = ax.bar(x - width/2, [expected[d] for d in x], width, label='Benford Expected',
               color='steelblue', alpha=0.7, edgecolor='navy')
bars2 = ax.bar(x + width/2, [observed[d] for d in x], width, label='Observed (Price Changes)',
               color='coral', alpha=0.7, edgecolor='darkred')

ax.set_xlabel('First Digit', fontsize=12)
ax.set_ylabel('Proportion', fontsize=12)
ax.set_title('Benford Distribution: Expected vs Observed', fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Deviation chart
deviations = {d: observed[d] - expected[d] for d in range(1, 10)}
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if v >= 0 else 'red' for v in deviations.values()]
ax.bar(deviations.keys(), deviations.values(), color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('First Digit')
ax.set_ylabel('Deviation from Benford')
ax.set_title('Deviation from Expected Benford Distribution')
ax.set_xticks(list(range(1, 10)))
plt.tight_layout()
plt.show()

## 5. Sliding Window Analysis

In [ ]:
# Run sliding window analysis
benford_result = run_benford_analysis(price_changes)
window_results = benford_result['window_results']
anomaly_windows = benford_result['anomaly_windows']

print(f'Total windows analyzed: {len(window_results)}')
print(f'Non-conforming windows: {len(anomaly_windows)}')

if window_results:
    # Plot chi-squared p-values over windows
    p_values = [w.get('chi_squared_p', None) for w in window_results]
    mad_values = [w.get('mad', None) for w in window_results]
    window_indices = [w['window_start_idx'] for w in window_results]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    ax = axes[0]
    ax.plot(window_indices, p_values, 'o-', markersize=3, color='steelblue')
    ax.axhline(y=0.05, color='red', linestyle='--', label='alpha=0.05')
    ax.set_ylabel('p-value')
    ax.set_title('Chi-Squared p-value per Window')
    ax.legend()
    
    ax = axes[1]
    ax.plot(window_indices, mad_values, 'o-', markersize=3, color='coral')
    ax.axhline(y=0.015, color='red', linestyle='--', label='Marginal threshold')
    ax.axhline(y=0.006, color='green', linestyle='--', label='Close conformity')
    ax.set_ylabel('MAD')
    ax.set_xlabel('Window Start Index')
    ax.set_title('MAD Conformity per Window')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Comparison: Natural vs Synthetic Data

In [ ]:
# Generate synthetic conforming (exponential) and non-conforming (uniform) data
np.random.seed(42)
exp_data = pd.Series(np.random.exponential(100, 500))
uniform_data = pd.Series(np.random.uniform(100, 999, 500))

exp_result = detector.analyze(exp_data)
uni_result = detector.analyze(uniform_data)
real_result = analysis

print('Comparison:')
print(f'{"Dataset":<25} {"Conforming":<12} {"Chi2 p":<12} {"MAD":<10} {"Level"}')
print('-' * 70)
print(f'{"Exponential (natural)":<25} {str(exp_result["overall_conforming"]):<12} '
      f'{exp_result["chi_squared"]["p_value"]:<12} '
      f'{exp_result["mad_test"]["mad"]:<10} {exp_result["mad_test"]["conformity_level"]}')
print(f'{"Uniform (fabricated)":<25} {str(uni_result["overall_conforming"]):<12} '
      f'{uni_result["chi_squared"]["p_value"]:<12} '
      f'{uni_result["mad_test"]["mad"]:<10} {uni_result["mad_test"]["conformity_level"]}')
print(f'{"Real price changes":<25} {str(real_result["overall_conforming"]):<12} '
      f'{real_result["chi_squared"]["p_value"]:<12} '
      f'{real_result["mad_test"]["mad"]:<10} {real_result["mad_test"]["conformity_level"]}')

---

## Summary

The Benford's Law detector analyzes first-digit distributions to identify
potentially fabricated data. Using price changes as a proxy (since trade
volume data is not yet available), the analysis reveals whether the market's
numerical patterns follow natural distributions or show signs of manipulation.

**Limitation**: Ideally this detector should analyze trade sizes/volumes.
Currently using price deltas as proxy until Member B's pipeline adds
trade data collection.